In [5]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from time import time
import threading, queue, concurrent.futures

# Clear top-level parameters
INPUT_VIDEO_PATH = r"videos\10 mins.mp4"
OUTPUT_VIDEO_PATH = "deflicker_output_r_1.mp4"
WINDOW_LENGTH = 16  # Number of frames per window
NUM_PROC_WORKERS = 4


class IdentityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Identity()
        print("[✓] Initialized identity model")
    def forward(self, x):
        return x

def process_video(input_path, output_path, window_length, num_proc_workers):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video specs: {width}x{height} @ {fps:.2f} FPS")
    
    # Queues for pipelining
    window_queue = queue.Queue(maxsize=100)
    output_queue = queue.Queue(maxsize=100)
    
    # Reader thread: groups frames into windows
    def reader():
        window = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            window.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if len(window) == window_length:
                window_queue.put((idx, np.array(window)))
                idx += 1
                window = []
        cap.release()
        for _ in range(num_proc_workers):
            window_queue.put(None)
    
    # Processing function: runs model on one window
    def process_window(item):
        index, window = item
        tensor = torch.from_numpy(window).float() / 255.0
        tensor = tensor.unsqueeze(0).to(device, non_blocking=True)
        with torch.no_grad():
            output = model(tensor).cpu().numpy()
        processed = (output * 255).astype(np.uint8).reshape(window_length, height, width, 3)
        return (index, processed)
    
    # Writer thread: writes windows in order
    def writer():
        next_idx = 0
        buffer = {}
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        while True:
            item = output_queue.get()
            if item is None:
                break
            index, proc_window = item
            buffer[index] = proc_window
            while next_idx in buffer:
                for frame in buffer.pop(next_idx):
                    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                next_idx += 1
        while next_idx in buffer:
            for frame in buffer.pop(next_idx):
                out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            next_idx += 1
        out.release()
        print("\nVideo saved successfully!")
    
    # Set device and model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = IdentityModel().to(device)
    model.eval()
    
    start_time = time()
    reader_thread = threading.Thread(target=reader)
    writer_thread = threading.Thread(target=writer)
    reader_thread.start()
    writer_thread.start()
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_proc_workers) as executor:
        futures = []
        while True:
            item = window_queue.get()
            if item is None:
                break
            futures.append(executor.submit(process_window, item))
        for future in concurrent.futures.as_completed(futures):
            output_queue.put(future.result())
    
    output_queue.put(None)
    reader_thread.join()
    writer_thread.join()
    print(f"\nTotal processing time: {time() - start_time:.2f} seconds")

if __name__ == '__main__':
    process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH, WINDOW_LENGTH, NUM_PROC_WORKERS)
    print("\nDownload processed video from:", OUTPUT_VIDEO_PATH)


Video specs: 1920x1080 @ 25.00 FPS
[✓] Initialized identity model


KeyboardInterrupt: 

Chunking for video input

In [ ]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from time import time, strftime, localtime
import threading, queue, concurrent.futures

# Top-level parameters
INPUT_VIDEO_PATH = r"videos\Example Video With Flickering (360p).mp4"
OUTPUT_VIDEO_PATH = "output\\" + INPUT_VIDEO_PATH
WINDOW_LENGTH = 16        # Number of frames per window
NUM_PROC_WORKERS = 4      # Number of parallel processing threads
MAX_ACTIVE_FUTURES = 50   # Maximum tasks to hold in RAM at once

def log(msg):
    print(f"[{strftime('%H:%M:%S', localtime())}] {msg}")

class IdentityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Identity()
        log("Initialized identity model")
    def forward(self, x):
        return x

def process_video(input_path, output_path, window_length, num_proc_workers, max_active_futures):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    log(f"Video specs: {width}x{height} @ {fps:.2f} FPS")
    
    # Queues for pipelining: window_queue for reading; output_queue for writing.
    window_queue = queue.Queue(maxsize=100)
    output_queue = queue.Queue(maxsize=100)
    
    # Reader thread: groups frames into windows
    def reader():
        window = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            window.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if len(window) == window_length:
                window_queue.put((idx, np.array(window)))
                if idx % 10 == 0:
                    log(f"Reader queued window {idx} (queue size: {window_queue.qsize()})")
                idx += 1
                window = []
        cap.release()
        log(f"Reader finished: total windows queued = {idx}")
        # Push one sentinel per worker so the processing loop knows when reading is done
        for _ in range(num_proc_workers):
            window_queue.put(None)
    
    # Processing function: runs model on one window
    def process_window(item):
        index, window = item
        tensor = torch.from_numpy(window).float() / 255.0
        tensor = tensor.unsqueeze(0).to(device, non_blocking=True)
        with torch.no_grad():
            output = model(tensor).cpu().numpy()
        processed = (output * 255).astype(np.uint8).reshape(window_length, height, width, 3)
        return (index, processed)
    
    # Writer thread: writes windows in order
    def writer():
        next_idx = 0
        buffer = {}
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        log("Writer started")
        while True:
            item = output_queue.get()
            if item is None:
                break
            index, proc_window = item
            buffer[index] = proc_window
            # Write windows in order as soon as possible
            while next_idx in buffer:
                out_window = buffer.pop(next_idx)
                for frame in out_window:
                    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                log(f"Writer wrote window {next_idx} (buffer size: {len(buffer)})")
                next_idx += 1
        # Flush any remaining frames
        while next_idx in buffer:
            for frame in buffer.pop(next_idx):
                out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            log(f"Writer flushed window {next_idx}")
            next_idx += 1
        out.release()
        log("Writer finished and video saved successfully!")
    
    # Set device and model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = IdentityModel().to(device)
    model.eval()
    
    start_time = time()
    log("Processing started")
    reader_thread = threading.Thread(target=reader)
    writer_thread = threading.Thread(target=writer)
    reader_thread.start()
    writer_thread.start()
    
    # Use a ThreadPoolExecutor to process windows concurrently while limiting the number of tasks in RAM.
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_proc_workers) as executor:
        active_futures = []
        finished_reading = False
        total_processed = 0
        # Loop until reading is finished and all futures are processed
        while not finished_reading or active_futures:
            # Fill the active_futures list until max_active_futures is reached
            while not finished_reading and len(active_futures) < max_active_futures:
                item = window_queue.get()
                if item is None:
                    finished_reading = True
                    log("No more windows to read")
                    break
                active_futures.append(executor.submit(process_window, item))
            if active_futures:
                done, pending = concurrent.futures.wait(
                    active_futures, return_when=concurrent.futures.FIRST_COMPLETED
                )
                for future in done:
                    output_queue.put(future.result())
                    total_processed += 1
                log(f"Processing batch: {total_processed} windows processed so far, {len(pending)} tasks pending")
                active_futures = list(pending)
    
    # Signal writer thread that processing is complete
    output_queue.put(None)
    reader_thread.join()
    writer_thread.join()
    elapsed = time() - start_time
    log(f"Total processing time: {elapsed:.2f} seconds")

if __name__ == '__main__':
    process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH, WINDOW_LENGTH, NUM_PROC_WORKERS, MAX_ACTIVE_FUTURES)
    print(f"\nDownload processed video from: {OUTPUT_VIDEO_PATH}")

Video specs: 1920x1080 @ 25.00 FPS
[✓] Initialized identity model


KeyboardInterrupt: 

Importing dask

In [3]:
!nvidia-smi


Sat Mar  8 11:34:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.127.08             Driver Version: 550.127.08     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070        Off |   00000000:65:00.0  On |                  N/A |
|  0%   39C    P8             18W /  270W |     808MiB /   8192MiB |     29%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
!pip install numpy torch opencv-python dask distributed dask_cuda

DASK - CUDA 

In [11]:
import cv2
import numpy as np
import torch
import torch.nn as nn
from time import time
import threading, queue
import os

# Dask imports
from dask.distributed import Client, wait

# Top-level parameters
INPUT_VIDEO_PATH = r"videos\10 mins.mp4"
OUTPUT_VIDEO_PATH = r"deflicker_output_dask_10mins.mp4"
WINDOW_LENGTH = 16        # Number of frames per window
NUM_PROC_WORKERS = 4      # Number of workers for processing (threads for non-cuda, processes for dask-cuda)
MAX_ACTIVE_FUTURES = 50   # Maximum tasks held in RAM at once
USE_DASK_CUDA = False     # Set to True to use dask-cuda (requires dask_cuda)

# Identity model
class IdentityModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Identity()
        print("[✓] Initialized identity model")

    def forward(self, x):
        return x

# Processing function
def process_window(item, model, device, height, width, window_length):
    index, window = item
    tensor = torch.from_numpy(window).float() / 255.0
    tensor = tensor.unsqueeze(0).to(device, non_blocking=True)
    with torch.no_grad():
        output = model(tensor).cpu().numpy()
    processed = (output * 255).astype(np.uint8).reshape(window_length, height, width, 3)
    return (index, processed)

# Main video processing function
def process_video(input_path, output_path, window_length, num_proc_workers, max_active_futures, use_dask_cuda):
    cap = cv2.VideoCapture(input_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f"Video specs: {width}x{height} @ {fps:.2f} FPS")

    # Queues for pipelining
    window_queue = queue.Queue(maxsize=100)
    output_queue = queue.Queue(maxsize=100)

    # Reader thread: groups frames into windows.
    def reader():
        window = []
        idx = 0
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            # Convert BGR to RGB.
            window.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if len(window) == window_length:
                window_queue.put((idx, np.array(window)))
                idx += 1
                window = []
        cap.release()
        for _ in range(num_proc_workers):
            window_queue.put(None)  # Sentinel values

    # Writer thread: writes processed windows in order.
    def writer():
        next_idx = 0
        buffer = {}
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
        while True:
            item = output_queue.get()
            if item is None:
                break
            index, proc_window = item
            buffer[index] = proc_window
            while next_idx in buffer:
                for frame in buffer.pop(next_idx):
                    out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
                next_idx += 1
        out.release()
        print("\n[✓] Video saved successfully!")

    # Set device and initialize model
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = IdentityModel().to(device)
    model.eval()

    # Create the Dask client
    if use_dask_cuda:
        from dask_cuda import LocalCUDACluster
        cluster = LocalCUDACluster(n_workers=num_proc_workers)
        client = Client(cluster)
    else:
        client = Client(processes=False, threads_per_worker=1, n_workers=num_proc_workers)

    if use_dask_cuda:
        scattered_model = client.scatter(model, broadcast=True)
        scattered_device = client.scatter(device, broadcast=True)
    else:
        scattered_model = model
        scattered_device = device

    start_time = time()
    reader_thread = threading.Thread(target=reader)
    writer_thread = threading.Thread(target=writer)
    reader_thread.start()
    writer_thread.start()

    # Dask processing loop
    active_futures = []
    finished_reading = False
    while not finished_reading or active_futures:
        while not finished_reading and len(active_futures) < max_active_futures:
            item = window_queue.get()
            if item is None:
                finished_reading = True
                break
            future = client.submit(
                process_window, item, scattered_model, scattered_device,
                height, width, window_length
            )
            active_futures.append(future)

        if active_futures:
            done, active_futures = wait(active_futures, return_when='FIRST_COMPLETED')
            for future in done:
                output_queue.put(future.result())

    # Signal writer thread that processing is complete.
    output_queue.put(None)
    reader_thread.join()
    writer_thread.join()
    client.close()

    # Confirm video was successfully saved
    if os.path.exists(output_path):
        print(f"\n[✓] Video processing complete! Download from: {os.path.abspath(output_path)}")
    else:
        print("\n[✗] Error: Processed video file was not found at the expected location!")
    
    print(f"\nTotal processing time: {time() - start_time:.2f} seconds")

if __name__ == '__main__':
    process_video(INPUT_VIDEO_PATH, OUTPUT_VIDEO_PATH, WINDOW_LENGTH,
                  NUM_PROC_WORKERS, MAX_ACTIVE_FUTURES, USE_DASK_CUDA)
    print("\nDownload processed video from:", OUTPUT_VIDEO_PATH)


Video specs: 0x0 @ 0.00 FPS
[✓] Initialized identity model

[✓] Video saved successfully!

[✗] Error: Processed video file was not found at the expected location!

Total processing time: 0.03 seconds

Download processed video from: deflicker_output_dask_10mins.mp4


In [12]:
import cv2
import os
import numpy as np
import torch
import dask
from dask.distributed import Client, as_completed
from queue import Queue
import threading

# ========== CONFIG ==========
input_path = "videos/10 mins.mp4"  
output_path = "deflicker_output_dask_10mins.mp4"  
window_length = 5  # Adjust based on model requirements
num_proc_workers = 4  # Dask parallel workers

# Initialize Dask Client
client = Client()

# Load model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.nn.Identity().to(device)  # Replace with actual model
model.eval()

# Open video and get properties
cap = cv2.VideoCapture(input_path)
if not cap.isOpened():
    print(f"[✗] Error: Unable to open video file '{input_path}'")
    exit()

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"[✓] Video specs: {width}x{height} @ {fps:.2f} FPS, Total Frames: {total_frames}")

# Queues for pipeline
window_queue = Queue()
output_queue = Queue()

# ========== READER THREAD ==========
def reader():
    window = []
    idx = 0
    total_read = 0  # Track frames read
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        total_read += 1
        window.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        if len(window) == window_length:
            window_queue.put((idx, np.array(window)))
            idx += 1
            window = []
    cap.release()
    print(f"[✓] Reader finished: {total_read} frames read.")
    
    # Signal processing to stop
    for _ in range(num_proc_workers):
        window_queue.put(None)

# ========== PROCESSING FUNCTION ==========
def process_window(item):
    if item is None:
        return None
    index, window = item
    print(f"[✓] Processing window {index}...")
    tensor = torch.from_numpy(window).float() / 255.0
    tensor = tensor.unsqueeze(0).to(device, non_blocking=True)
    with torch.no_grad():
        output = model(tensor).cpu().numpy()
    processed = (output * 255).astype(np.uint8).reshape(window_length, height, width, 3)
    print(f"[✓] Processed window {index}.")
    return (index, processed)

# ========== PROCESSING THREAD ==========
def processor():
    futures = []
    while True:
        item = window_queue.get()
        if item is None:
            break
        future = client.submit(process_window, item)
        futures.append(future)

    # Collect processed results
    for future in as_completed(futures):
        result = future.result()
        if result:
            output_queue.put(result)

# ========== WRITER THREAD ==========
def writer():
    next_idx = 0
    buffer = {}
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    if not out.isOpened():
        print("[✗] Error: VideoWriter failed to open output file.")
        return

    print("[✓] Writer initialized.")
    
    while True:
        item = output_queue.get()
        if item is None:
            break
        index, proc_window = item
        buffer[index] = proc_window
        while next_idx in buffer:
            for frame in buffer.pop(next_idx):
                out.write(cv2.cvtColor(frame, cv2.COLOR_RGB2BGR))
            next_idx += 1

    out.release()
    print(f"[✓] Video saved successfully at: {output_path}")

# ========== THREAD START ==========
reader_thread = threading.Thread(target=reader)
processor_thread = threading.Thread(target=processor)
writer_thread = threading.Thread(target=writer)

reader_thread.start()
processor_thread.start()
writer_thread.start()

reader_thread.join()
processor_thread.join()
output_queue.put(None)  # Signal writer to stop
writer_thread.join()

# Verify output file
if os.path.exists(output_path):
    print(f"[✓] Output file exists: {output_path} ({os.path.getsize(output_path) / (1024*1024):.2f} MB)")
else:
    print("[✗] Error: Processed video file was not found at the expected location!")


[✓] Video specs: 1920x1080 @ 25.00 FPS, Total Frames: 15208
[✓] Writer initialized.


/home/ailab/anaconda3/envs/SCTF/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 29.67 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


[✓] Processing window 0...
[✓] Processing window 1...
[✓] Processing window 5...
[✓] Processing window 6...
[✓] Processing window 10...
[✓] Processing window 11...
[✓] Processing window 2...
[✓] Processing window 7...
[✓] Processing window 12...
[✓] Processing window 3...
[✓] Processing window 8...
[✓] Processing window 13...
[✓] Processing window 4...
[✓] Processing window 9...
[✓] Processing window 14...
[✓] Processing window 15...
[✓] Processing window 16...
[✓] Processing window 17...
[✓] Processed window 12.
[✓] Processed window 4.
[✓] Processed window 3.
[✓] Processed window 6.
[✓] Processing window 18...
[✓] Processed window 0.
[✓] Processed window 7.
[✓] Processed window 15.
[✓] Processed window 13.
[✓] Processed window 1.
[✓] Processed window 9.
[✓] Processed window 2.
[✓] Processed window 10.
[✓] Processed window 8.
[✓] Processed window 14.
[✓] Processed window 11.
[✓] Processed window 5.
[✓] Processed window 17.
[✓] Processed window 16.
[✓] Processing window 19...
[✓] Proces

Spark